# Knee Joint Biomechanics Analysis - Mechanical Axis Calculation

This notebook analyzes motion capture data from knee joint trials to:
- Extract and process marker trajectories
- Calculate anatomical landmarks and coordinate systems
- Compute the mechanical axis of the lower limb
- Visualize joint kinematics and axes
- Analyze dynamic trials (rotation, bending, varus motion)

## Marker Definitions
- **Femur markers**: uFH (upper femoral head), FH (femoral head), MC (medial condyle), uFD/lFD (upper/lower femoral digitized), MFEC/LFEC (medial/lateral femoral epicondyle)
- **Tibia markers**: LTC/MTC (lateral/medial tibial condyle), uTD/lTD (upper/lower tibial digitized)
- **Foot markers**: Heel, MM/LM (medial/lateral malleolus), Toe

In [22]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation
from scipy.optimize import least_squares
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Data Loading and Preprocessing Functions

In [23]:
# Import required libraries
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation
from scipy.optimize import least_squares
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

def load_mocap_csv(filepath, skip_device_rows=True):
    """
    Robust loader for motion capture CSV files with multi-row headers.

    This function detects the header row containing 'Frame' (and 'Sub Frame')
    and, if present, will use the marker-name row above it together with the
    Frame row as a two-row header. It flattens MultiIndex columns into a
    single-level format like 'Subject 1:uFH:X'. It also removes a units row
    (e.g., rows containing 'mm' or 'N') when present and converts numeric
    columns to floats while preserving 'Frame' and 'Sub Frame'.

    Parameters:
    -----------
    filepath : str
        Path to the CSV file
    skip_device_rows : bool
        Deprecated: kept for API compatibility (detection is automatic)

    Returns:
    --------
    df : pandas DataFrame
        Loaded motion capture data with flattened column names
    """
    # Try to detect the header row (where column names like 'Frame' appear)
    header_row = None
    marker_name_row = None
    try:
        with open(filepath, 'r', encoding='latin-1') as f:
            # Read the first 40 lines to detect header patterns
            lines = [next(f) for _ in range(40)]
    except Exception:
        lines = []

    for i, line in enumerate(lines):
        if 'Frame' in line and 'Sub Frame' in line:
            header_row = i
            marker_name_row = i - 1 if i > 0 else None
            break

    try:
        if header_row is not None and marker_name_row is not None:
            # Use two-row header (marker names row + axis row) to create MultiIndex
            df = pd.read_csv(filepath, header=[marker_name_row, header_row], encoding='latin-1')

            # Flatten MultiIndex columns into 'marker:axis' (or single name for Frame)
            if isinstance(df.columns, pd.MultiIndex):
                new_cols = []
                for a, b in df.columns:
                    a_str = '' if pd.isna(a) else str(a).strip()
                    b_str = '' if pd.isna(b) else str(b).strip()
                    if a_str and b_str:
                        new_cols.append(f"{a_str}:{b_str}")
                    else:
                        new_cols.append(a_str or b_str)
                df.columns = new_cols
        elif skip_device_rows:
            # Fallback to previous behavior (skip first two rows)
            df = pd.read_csv(filepath, skiprows=2, encoding='latin-1')
        else:
            df = pd.read_csv(filepath, encoding='latin-1')
    except Exception:
        # If any parsing fails, fallback to a simple read
        df = pd.read_csv(filepath, encoding='latin-1')

    # If the first data row contains units like 'mm' or 'N', drop it
    try:
        if df.shape[0] > 0 and df.iloc[0].astype(str).str.contains('mm|N').any():
            df = df.iloc[1:].reset_index(drop=True)
    except Exception:
        pass

    # Convert columns to numeric where appropriate, keep Frame/Sub Frame as-is
    for col in df.columns:
        if str(col).strip() in ['Frame', 'Sub Frame']:
            continue
        # Try to coerce to numeric; if fails keep as-is
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception:
            pass

    return df


def extract_marker_trajectory(df, marker_name):
    """
    Extract X, Y, Z coordinates for a specific marker.
    
    Parameters:
    -----------
    df : pandas DataFrame
        Motion capture data
    marker_name : str
        Name of the marker (e.g., 'FH', 'MFEC')
    
    Returns:
    --------
    trajectory : numpy array (N x 3)
        X, Y, Z coordinates for each frame
    """
    # Find columns containing the marker name
    cols = [col for col in df.columns if marker_name in str(col)]
    
    if len(cols) < 3:
        print(f"Warning: Marker {marker_name} not found or incomplete")
        return None
    
    # Get first three columns for this marker (X, Y, Z)
    idx = df.columns.get_loc(cols[0])
    xyz_cols = df.columns[idx:idx+3]
    
    trajectory = df[xyz_cols].values.astype(float)
    
    return trajectory


def get_all_markers(df):
    """
    Extract all unique marker names from the dataframe using regex.

    This looks for patterns like 'Subject 1:FH:X' or 'FH:X' and extracts 'FH'.
    It ignores unnamed header tokens and non-axis columns like 'Frame' or 'Sub Frame'.
    """
    markers = []
    pattern = re.compile(r"(?:(?:Subject\s*\d+)?:)?(?P<marker>[A-Za-z0-9_]+):[XYZ]$")
    for col in df.columns:
        s = str(col).strip()
        m = pattern.search(s)
        if m:
            candidate = m.group('marker')
            if candidate not in markers and candidate not in ['Frame', 'Sub Frame', 'X', 'Y', 'Z', 'mm']:
                markers.append(candidate)
    return markers


## 2. Anatomical Coordinate System Functions

In [24]:
def normalize_vectors(v):
    """
    Normalize vectors along rows.
    """
    norms = np.linalg.norm(v, axis=-1, keepdims=True)
    norms[norms < 1e-10] = 1.0  # Avoid division by zero
    return v / norms


def build_coordinate_frame(origin, point_x, point_y):
    """
    Build a right-handed coordinate frame.
    
    Parameters:
    -----------
    origin : array (3,) or (N, 3)
        Origin of the coordinate system
    point_x : array (3,) or (N, 3)
        Point defining the X-axis direction
    point_y : array (3,) or (N, 3)
        Point defining the Y-axis direction
    
    Returns:
    --------
    R : array (3, 3) or (N, 3, 3)
        Rotation matrix representing the coordinate frame
    O : array (3,) or (N, 3)
        Origin position
    """
    # X-axis
    x_axis = normalize_vectors(point_x - origin)
    
    # Temporary Y direction
    y_temp = normalize_vectors(point_y - origin)
    
    # Z-axis (perpendicular to X and Y)
    z_axis = normalize_vectors(np.cross(x_axis, y_temp))
    
    # Recompute Y-axis to ensure orthogonality
    y_axis = normalize_vectors(np.cross(z_axis, x_axis))
    
    if x_axis.ndim == 1:
        R = np.column_stack([x_axis, y_axis, z_axis])
    else:
        R = np.stack([x_axis, y_axis, z_axis], axis=-1)
    
    return R, origin


def sphere_fit(points):
    """
    Fit a sphere to 3D points using least squares optimization.
    Used for finding joint centers (e.g., femoral head).
    
    Parameters:
    -----------
    points : array (N, 3)
        3D points to fit sphere to
    
    Returns:
    --------
    center : array (3,)
        Center of the fitted sphere
    radius : float
        Radius of the fitted sphere
    """
    def residuals(params, points):
        center = params[:3]
        radius = params[3]
        return np.linalg.norm(points - center, axis=1) - radius
    
    # Initial guess: centroid and average distance
    centroid = np.mean(points, axis=0)
    radius_init = np.mean(np.linalg.norm(points - centroid, axis=1))
    x0 = np.append(centroid, radius_init)
    
    result = least_squares(residuals, x0, args=(points,))
    
    center = result.x[:3]
    radius = result.x[3]
    
    return center, radius

## 3. Mechanical Axis Calculation Functions

In [25]:
def calculate_hip_center(femoral_head_trajectory):
    """
    Calculate hip joint center from femoral head marker trajectory.
    If the trajectory shows movement, fit a sphere to estimate the center.
    
    Parameters:
    -----------
    femoral_head_trajectory : array (N, 3)
        Trajectory of femoral head marker
    
    Returns:
    --------
    hip_center : array (3,)
        Estimated hip joint center
    """
    # Remove NaN values
    valid_points = femoral_head_trajectory[~np.isnan(femoral_head_trajectory).any(axis=1)]
    
    if len(valid_points) < 10:
        # Not enough points, use mean
        hip_center = np.nanmean(femoral_head_trajectory, axis=0)
    else:
        # Check if there's significant movement
        movement_range = np.ptp(valid_points, axis=0)
        
        if np.max(movement_range) > 20:  # More than 20mm movement
            # Fit sphere to find center of rotation
            hip_center, _ = sphere_fit(valid_points)
        else:
            # Static or minimal movement, use mean
            hip_center = np.mean(valid_points, axis=0)
    
    return hip_center


def calculate_knee_center(medial_condyle, lateral_condyle, 
                          medial_epicondyle=None, lateral_epicondyle=None):
    """
    Calculate knee joint center as midpoint between condyles or epicondyles.
    
    Parameters:
    -----------
    medial_condyle : array (N, 3)
        Medial condyle marker trajectory
    lateral_condyle : array (N, 3)
        Lateral condyle marker trajectory
    medial_epicondyle : array (N, 3), optional
        Medial epicondyle marker trajectory
    lateral_epicondyle : array (N, 3), optional
        Lateral epicondyle marker trajectory
    
    Returns:
    --------
    knee_center : array (N, 3)
        Knee joint center trajectory
    """
    # Prefer epicondyles if available (more accurate for joint center)
    if medial_epicondyle is not None and lateral_epicondyle is not None:
        knee_center = (medial_epicondyle + lateral_epicondyle) / 2.0
    else:
        knee_center = (medial_condyle + lateral_condyle) / 2.0
    
    return knee_center


def calculate_ankle_center(medial_malleolus, lateral_malleolus):
    """
    Calculate ankle joint center as midpoint between malleoli.
    
    Parameters:
    -----------
    medial_malleolus : array (N, 3)
        Medial malleolus marker trajectory
    lateral_malleolus : array (N, 3)
        Lateral malleolus marker trajectory
    
    Returns:
    --------
    ankle_center : array (N, 3)
        Ankle joint center trajectory
    """
    ankle_center = (medial_malleolus + lateral_malleolus) / 2.0
    return ankle_center


def calculate_mechanical_axis(hip_center, knee_center, ankle_center):
    """
    Calculate the mechanical axis of the lower limb.
    
    The mechanical axis is the line connecting the hip center to the ankle center.
    In normal alignment, it should pass near or through the knee center.
    
    Parameters:
    -----------
    hip_center : array (3,) or (N, 3)
        Hip joint center
    knee_center : array (3,) or (N, 3)
        Knee joint center
    ankle_center : array (3,) or (N, 3)
        Ankle joint center
    
    Returns:
    --------
    mechanical_axis : dict
        Dictionary containing:
        - 'axis_vector': Unit vector from hip to ankle
        - 'axis_length': Distance from hip to ankle
        - 'knee_offset': Perpendicular distance from knee to mechanical axis
        - 'varus_valgus_angle': Angle between femoral and tibial axes (degrees)
    """
    # Ensure arrays are 1D if single frame
    if hip_center.ndim == 1:
        hip_center = hip_center.reshape(1, -1)
        knee_center = knee_center.reshape(1, -1)
        ankle_center = ankle_center.reshape(1, -1)
        single_frame = True
    else:
        single_frame = False
    
    # Mechanical axis vector (hip to ankle)
    axis_vector_full = ankle_center - hip_center
    axis_length = np.linalg.norm(axis_vector_full, axis=-1)
    axis_vector = normalize_vectors(axis_vector_full)
    
    # Calculate knee offset (perpendicular distance from knee to mechanical axis)
    hip_to_knee = knee_center - hip_center
    projection = np.sum(hip_to_knee * axis_vector, axis=-1, keepdims=True) * axis_vector
    knee_offset_vector = hip_to_knee - projection
    knee_offset = np.linalg.norm(knee_offset_vector, axis=-1)
    
    # Calculate varus/valgus angle (femoral axis to tibial axis)
    femoral_axis = normalize_vectors(knee_center - hip_center)
    tibial_axis = normalize_vectors(ankle_center - knee_center)
    
    # Angle between axes
    dot_product = np.sum(femoral_axis * tibial_axis, axis=-1)
    dot_product = np.clip(dot_product, -1.0, 1.0)
    varus_valgus_angle = np.rad2deg(np.arccos(dot_product))
    
    result = {
        'axis_vector': axis_vector[0] if single_frame else axis_vector,
        'axis_length': axis_length[0] if single_frame else axis_length,
        'knee_offset': knee_offset[0] if single_frame else knee_offset,
        'knee_offset_vector': knee_offset_vector[0] if single_frame else knee_offset_vector,
        'varus_valgus_angle': varus_valgus_angle[0] if single_frame else varus_valgus_angle,
        'femoral_axis': femoral_axis[0] if single_frame else femoral_axis,
        'tibial_axis': tibial_axis[0] if single_frame else tibial_axis
    }
    
    return result

## 4. Visualization Functions

In [26]:
def plot_3d_skeleton(hip_center, knee_center, ankle_center, 
                     mechanical_axis_info=None,
                     markers_dict=None,
                     frame_idx=0,
                     title="Lower Limb Skeleton"):
    """
    Plot 3D visualization of the lower limb skeleton with mechanical axis.
    
    Parameters:
    -----------
    hip_center : array (3,) or (N, 3)
        Hip joint center
    knee_center : array (3,) or (N, 3)
        Knee joint center
    ankle_center : array (3,) or (N, 3)
        Ankle joint center
    mechanical_axis_info : dict, optional
        Information about mechanical axis from calculate_mechanical_axis()
    markers_dict : dict, optional
        Dictionary of marker trajectories to plot
    frame_idx : int
        Frame index to plot (for dynamic trials)
    title : str
        Plot title
    """
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Extract frame data
    if hip_center.ndim > 1:
        hip = hip_center[frame_idx]
        knee = knee_center[frame_idx]
        ankle = ankle_center[frame_idx]
    else:
        hip = hip_center
        knee = knee_center
        ankle = ankle_center
    
    # Plot bones (femur and tibia)
    ax.plot([hip[0], knee[0]], [hip[1], knee[1]], [hip[2], knee[2]], 
            'b-', linewidth=4, label='Femur')
    ax.plot([knee[0], ankle[0]], [knee[1], ankle[1]], [knee[2], ankle[2]], 
            'g-', linewidth=4, label='Tibia')
    
    # Plot joints
    ax.scatter(*hip, color='red', s=200, marker='o', label='Hip Center', edgecolors='black', linewidths=2)
    ax.scatter(*knee, color='yellow', s=200, marker='o', label='Knee Center', edgecolors='black', linewidths=2)
    ax.scatter(*ankle, color='orange', s=200, marker='o', label='Ankle Center', edgecolors='black', linewidths=2)
    
    # Plot mechanical axis
    ax.plot([hip[0], ankle[0]], [hip[1], ankle[1]], [hip[2], ankle[2]], 
            'r--', linewidth=2, label='Mechanical Axis', alpha=0.7)
    
    # Plot knee offset if available
    if mechanical_axis_info is not None:
        if 'knee_offset_vector' in mechanical_axis_info:
            offset_vec = mechanical_axis_info['knee_offset_vector']
            if offset_vec.ndim > 1:
                offset_vec = offset_vec[frame_idx]
            
            # Point on mechanical axis closest to knee
            axis_vec = mechanical_axis_info['axis_vector']
            if axis_vec.ndim > 1:
                axis_vec = axis_vec[frame_idx]
            
            hip_to_knee = knee - hip
            projection_length = np.dot(hip_to_knee, axis_vec)
            closest_point = hip + projection_length * axis_vec
            
            ax.plot([knee[0], closest_point[0]], 
                   [knee[1], closest_point[1]], 
                   [knee[2], closest_point[2]], 
                   'm-', linewidth=2, label=f'Knee Offset: {mechanical_axis_info["knee_offset"]:.1f} mm')
    
    # Plot additional markers if provided
    if markers_dict is not None:
        for marker_name, trajectory in markers_dict.items():
            if trajectory is not None and not np.isnan(trajectory[frame_idx]).any():
                ax.scatter(*trajectory[frame_idx], s=50, alpha=0.6, label=marker_name)
    
    # Set labels and title
    ax.set_xlabel('X (mm)', fontsize=12)
    ax.set_ylabel('Y (mm)', fontsize=12)
    ax.set_zlabel('Z (mm)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Equal aspect ratio
    max_range = np.array([hip[0]-ankle[0], hip[1]-ankle[1], hip[2]-ankle[2]]).max() / 2.0
    mid_x = (hip[0] + ankle[0]) * 0.5
    mid_y = (hip[1] + ankle[1]) * 0.5
    mid_z = (hip[2] + ankle[2]) * 0.5
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    ax.legend(loc='upper left', fontsize=8)
    ax.view_init(elev=20, azim=45)
    
    plt.tight_layout()
    return fig


def plot_angle_time_series(mechanical_axis_info, fps=100, title="Knee Joint Angles"):
    """
    Plot time series of joint angles.
    
    Parameters:
    -----------
    mechanical_axis_info : dict
        Information from calculate_mechanical_axis()
    fps : float
        Frames per second of the trial
    title : str
        Plot title
    """
    varus_valgus = mechanical_axis_info['varus_valgus_angle']
    knee_offset = mechanical_axis_info['knee_offset']
    
    if np.isscalar(varus_valgus):
        print("Single frame data - no time series to plot")
        return
    
    time = np.arange(len(varus_valgus)) / fps
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Varus/Valgus angle
    ax1.plot(time, varus_valgus, 'b-', linewidth=2)
    ax1.axhline(y=180, color='r', linestyle='--', alpha=0.5, label='Perfect Alignment (180°)')
    ax1.set_ylabel('Angle (degrees)', fontsize=12)
    ax1.set_title('Femur-Tibia Angle (Varus/Valgus)', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Knee offset
    ax2.plot(time, knee_offset, 'g-', linewidth=2)
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero Offset')
    ax2.set_xlabel('Time (s)', fontsize=12)
    ax2.set_ylabel('Offset (mm)', fontsize=12)
    ax2.set_title('Knee Offset from Mechanical Axis', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    return fig


def plot_2d_frontal_view(hip_center, knee_center, ankle_center,
                         mechanical_axis_info=None,
                         frame_idx=0,
                         title="Frontal Plane View"):
    """
    Plot 2D frontal plane view (coronal plane) of the lower limb.
    
    Parameters:
    -----------
    hip_center : array
        Hip joint center
    knee_center : array
        Knee joint center
    ankle_center : array
        Ankle joint center
    mechanical_axis_info : dict
        Mechanical axis information
    frame_idx : int
        Frame to plot
    title : str
        Plot title
    """
    fig, ax = plt.subplots(1, 1, figsize=(8, 12))
    
    # Extract frame data
    if hip_center.ndim > 1:
        hip = hip_center[frame_idx]
        knee = knee_center[frame_idx]
        ankle = ankle_center[frame_idx]
    else:
        hip = hip_center
        knee = knee_center
        ankle = ankle_center
    
    # Use X and Z coordinates for frontal view (assuming Y is forward)
    ax.plot([hip[0], knee[0]], [hip[2], knee[2]], 'b-', linewidth=8, label='Femur')
    ax.plot([knee[0], ankle[0]], [knee[2], ankle[2]], 'g-', linewidth=8, label='Tibia')
    
    # Mechanical axis
    ax.plot([hip[0], ankle[0]], [hip[2], ankle[2]], 'r--', linewidth=3, label='Mechanical Axis')
    
    # Joints
    ax.scatter(hip[0], hip[2], color='red', s=300, marker='o', 
              label='Hip', edgecolors='black', linewidths=2, zorder=5)
    ax.scatter(knee[0], knee[2], color='yellow', s=300, marker='o', 
              label='Knee', edgecolors='black', linewidths=2, zorder=5)
    ax.scatter(ankle[0], ankle[2], color='orange', s=300, marker='o', 
              label='Ankle', edgecolors='black', linewidths=2, zorder=5)
    
    # Add angle annotation
    if mechanical_axis_info is not None:
        angle = mechanical_axis_info['varus_valgus_angle']
        if not np.isscalar(angle):
            angle = angle[frame_idx]
        offset = mechanical_axis_info['knee_offset']
        if not np.isscalar(offset):
            offset = offset[frame_idx]
        
        ax.text(0.05, 0.95, f'Varus/Valgus Angle: {angle:.1f}°\nKnee Offset: {offset:.1f} mm',
               transform=ax.transAxes, fontsize=12,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_xlabel('Medial-Lateral (mm)', fontsize=12)
    ax.set_ylabel('Superior-Inferior (mm)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.axis('equal')
    ax.invert_yaxis()  # Invert to have superior at top
    
    plt.tight_layout()
    return fig

## 5. Load and Process Data

Now let's load your data files and perform the analysis.

In [27]:
# Update these paths to match your file locations
DATA_DIR = './data/'  # Change this to your data directory

# Load static trial
print("Loading static trial...")
static_trial = load_mocap_csv(DATA_DIR + 'Static_Trial_Bone_Model.csv')
print(f"Static trial shape: {static_trial.shape}")

# Get available markers
markers = get_all_markers(static_trial)
print(f"\nAvailable markers: {markers}")

Loading static trial...
Static trial shape: (578, 47)

Available markers: ['uFH', '3_level_0', '4_level_0', 'FH', '6_level_0', '7_level_0', 'MC', '9_level_0', '10_level_0', 'uFD', '12_level_0', '13_level_0', 'lFD', '15_level_0', '16_level_0', 'MFEC', '18_level_0', '19_level_0', 'LFEC', '21_level_0', '22_level_0', 'LTC', '24_level_0', '25_level_0', 'MTC', '27_level_0', '28_level_0', 'uTD', '30_level_0', '31_level_0', 'lTD', '33_level_0', '34_level_0', 'Heel', '36_level_0', '37_level_0', 'MM', '39_level_0', '40_level_0', 'LM', '42_level_0', '43_level_0', 'Toe', '45_level_0', '46_level_0']


In [28]:
print(static_trial.columns.tolist()[:60])  # show the first ~60 columns

['Unnamed: 0_level_0:Frame', 'Unnamed: 1_level_0:Sub Frame', 'Subject 1:uFH:X', 'Unnamed: 3_level_0:Y', 'Unnamed: 4_level_0:Z', 'Subject 1:FH:X', 'Unnamed: 6_level_0:Y', 'Unnamed: 7_level_0:Z', 'Subject 1:MC:X', 'Unnamed: 9_level_0:Y', 'Unnamed: 10_level_0:Z', 'Subject 1:uFD:X', 'Unnamed: 12_level_0:Y', 'Unnamed: 13_level_0:Z', 'Subject 1:lFD:X', 'Unnamed: 15_level_0:Y', 'Unnamed: 16_level_0:Z', 'Subject 1:MFEC:X', 'Unnamed: 18_level_0:Y', 'Unnamed: 19_level_0:Z', 'Subject 1:LFEC:X', 'Unnamed: 21_level_0:Y', 'Unnamed: 22_level_0:Z', 'Subject 1:LTC:X', 'Unnamed: 24_level_0:Y', 'Unnamed: 25_level_0:Z', 'Subject 1:MTC:X', 'Unnamed: 27_level_0:Y', 'Unnamed: 28_level_0:Z', 'Subject 1:uTD:X', 'Unnamed: 30_level_0:Y', 'Unnamed: 31_level_0:Z', 'Subject 1:lTD:X', 'Unnamed: 33_level_0:Y', 'Unnamed: 34_level_0:Z', 'Subject 1:Heel:X', 'Unnamed: 36_level_0:Y', 'Unnamed: 37_level_0:Z', 'Subject 1:MM:X', 'Unnamed: 39_level_0:Y', 'Unnamed: 40_level_0:Z', 'Subject 1:LM:X', 'Unnamed: 42_level_0:Y', 'Unn

In [29]:
print(get_all_markers(static_trial))

['uFH', '3_level_0', '4_level_0', 'FH', '6_level_0', '7_level_0', 'MC', '9_level_0', '10_level_0', 'uFD', '12_level_0', '13_level_0', 'lFD', '15_level_0', '16_level_0', 'MFEC', '18_level_0', '19_level_0', 'LFEC', '21_level_0', '22_level_0', 'LTC', '24_level_0', '25_level_0', 'MTC', '27_level_0', '28_level_0', 'uTD', '30_level_0', '31_level_0', 'lTD', '33_level_0', '34_level_0', 'Heel', '36_level_0', '37_level_0', 'MM', '39_level_0', '40_level_0', 'LM', '42_level_0', '43_level_0', 'Toe', '45_level_0', '46_level_0']


## 6. Extract Marker Trajectories from Static Trial

In [30]:
# Extract key markers for static trial
print("Extracting marker trajectories...\n")

# Femur markers
FH = extract_marker_trajectory(static_trial, 'FH')  # Femoral head
uFH = extract_marker_trajectory(static_trial, 'uFH')  # Upper femoral head
MFEC = extract_marker_trajectory(static_trial, 'MFEC')  # Medial femoral epicondyle
LFEC = extract_marker_trajectory(static_trial, 'LFEC')  # Lateral femoral epicondyle
MC = extract_marker_trajectory(static_trial, 'MC')  # Medial condyle

# Tibia markers
MTC = extract_marker_trajectory(static_trial, 'MTC')  # Medial tibial condyle
LTC = extract_marker_trajectory(static_trial, 'LTC')  # Lateral tibial condyle

# Ankle markers
MM = extract_marker_trajectory(static_trial, 'MM')  # Medial malleolus
LM = extract_marker_trajectory(static_trial, 'LM')  # Lateral malleolus

# Other markers
Heel = extract_marker_trajectory(static_trial, 'Heel')
Toe = extract_marker_trajectory(static_trial, 'Toe')

# Print info about extracted markers
marker_dict = {
    'FH': FH, 'uFH': uFH, 'MFEC': MFEC, 'LFEC': LFEC, 'MC': MC,
    'MTC': MTC, 'LTC': LTC, 'MM': MM, 'LM': LM, 'Heel': Heel, 'Toe': Toe
}

for name, traj in marker_dict.items():
    if traj is not None:
        print(f"{name}: {traj.shape}, Mean position: [{traj[0,0]:.1f}, {traj[0,1]:.1f}, {traj[0,2]:.1f}] mm")
    else:
        print(f"{name}: Not found")

Extracting marker trajectories...

FH: Not found
uFH: Not found
MFEC: Not found
LFEC: Not found
MC: Not found
MTC: Not found
LTC: Not found
MM: Not found
LM: Not found
Heel: Not found
Toe: Not found


## 7. Calculate Joint Centers and Mechanical Axis (Static Trial)

In [31]:
# Calculate hip center from femoral head
if FH is not None:
    hip_center_static = calculate_hip_center(FH)
    print(f"Hip Center: [{hip_center_static[0]:.1f}, {hip_center_static[1]:.1f}, {hip_center_static[2]:.1f}] mm")
else:
    print("Warning: Femoral head marker not found")

# Calculate knee center
if MFEC is not None and LFEC is not None:
    knee_center_static = calculate_knee_center(MC, LTC, MFEC, LFEC)
    print(f"Knee Center: [{knee_center_static[0,0]:.1f}, {knee_center_static[0,1]:.1f}, {knee_center_static[0,2]:.1f}] mm")
elif MTC is not None and LTC is not None:
    knee_center_static = calculate_knee_center(MTC, LTC)
    print(f"Knee Center: [{knee_center_static[0,0]:.1f}, {knee_center_static[0,1]:.1f}, {knee_center_static[0,2]:.1f}] mm")
else:
    print("Warning: Knee markers not found")

# Calculate ankle center
if MM is not None and LM is not None:
    ankle_center_static = calculate_ankle_center(MM, LM)
    print(f"Ankle Center: [{ankle_center_static[0,0]:.1f}, {ankle_center_static[0,1]:.1f}, {ankle_center_static[0,2]:.1f}] mm")
else:
    print("Warning: Ankle markers not found")

# Calculate mechanical axis
print("\n" + "="*60)
print("MECHANICAL AXIS ANALYSIS - STATIC TRIAL")
print("="*60)

mech_axis_static = calculate_mechanical_axis(
    hip_center_static,
    knee_center_static[0],  # Take first frame
    ankle_center_static[0]  # Take first frame
)

print(f"\nMechanical Axis Length: {mech_axis_static['axis_length']:.1f} mm")
print(f"Knee Offset from Axis: {mech_axis_static['knee_offset']:.1f} mm")
print(f"Varus/Valgus Angle: {mech_axis_static['varus_valgus_angle']:.1f}°")
print(f"  (180° = perfect alignment, >180° = valgus, <180° = varus)")

# Interpretation
if abs(180 - mech_axis_static['varus_valgus_angle']) < 5:
    alignment = "Normal"
elif mech_axis_static['varus_valgus_angle'] > 180:
    alignment = "Valgus (knock-knee)"
else:
    alignment = "Varus (bow-legged)"

print(f"\nAlignment Assessment: {alignment}")
print(f"Knee offset interpretation: {'Normal (<20mm)' if mech_axis_static['knee_offset'] < 20 else 'Elevated (>20mm)'}")


MECHANICAL AXIS ANALYSIS - STATIC TRIAL


NameError: name 'hip_center_static' is not defined

## 8. Visualize Static Trial Results

In [ ]:
# 3D visualization
fig1 = plot_3d_skeleton(
    hip_center_static,
    knee_center_static[0],
    ankle_center_static[0],
    mechanical_axis_info=mech_axis_static,
    markers_dict={'MFEC': MFEC, 'LFEC': LFEC, 'MM': MM, 'LM': LM},
    frame_idx=0,
    title="Static Trial - Lower Limb Mechanical Axis"
)
plt.show()

# 2D frontal view
fig2 = plot_2d_frontal_view(
    hip_center_static,
    knee_center_static[0],
    ankle_center_static[0],
    mechanical_axis_info=mech_axis_static,
    frame_idx=0,
    title="Static Trial - Frontal Plane (Coronal) View"
)
plt.show()

NameError: name 'hip_center_static' is not defined

## 9. Analyze Dynamic Trials

Now let's analyze the dynamic trials to see how the mechanical axis changes during movement.

In [ ]:
# Load dynamic rotation trial
print("Loading dynamic rotation trial...")
dynamic_rotation = load_mocap_csv(DATA_DIR + 'dynamic__trial_rotation_1.csv')
print(f"Dynamic rotation shape: {dynamic_rotation.shape}")

# Extract markers from dynamic trial
print("\nExtracting markers from dynamic trial...")
FH_dyn = extract_marker_trajectory(dynamic_rotation, 'FH')
MFEC_dyn = extract_marker_trajectory(dynamic_rotation, 'MFEC')
LFEC_dyn = extract_marker_trajectory(dynamic_rotation, 'LFEC')
MTC_dyn = extract_marker_trajectory(dynamic_rotation, 'MTC')
LTC_dyn = extract_marker_trajectory(dynamic_rotation, 'LTC')
MM_dyn = extract_marker_trajectory(dynamic_rotation, 'MM')
LM_dyn = extract_marker_trajectory(dynamic_rotation, 'LM')

# Calculate joint centers for dynamic trial
if FH_dyn is not None:
    hip_center_dyn = calculate_hip_center(FH_dyn)
    print(f"Hip center calculated from {len(FH_dyn)} frames")

if MFEC_dyn is not None and LFEC_dyn is not None:
    knee_center_dyn = calculate_knee_center(MTC_dyn, LTC_dyn, MFEC_dyn, LFEC_dyn)
else:
    knee_center_dyn = calculate_knee_center(MTC_dyn, LTC_dyn)

ankle_center_dyn = calculate_ankle_center(MM_dyn, LM_dyn)

# Calculate mechanical axis for all frames
print("\nCalculating mechanical axis for dynamic trial...")
mech_axis_dyn = calculate_mechanical_axis(
    hip_center_dyn,
    knee_center_dyn,
    ankle_center_dyn
)

print(f"\nDynamic Trial Analysis Summary:")
print(f"Number of frames: {len(knee_center_dyn)}")
print(f"Mean varus/valgus angle: {np.mean(mech_axis_dyn['varus_valgus_angle']):.1f}° ± {np.std(mech_axis_dyn['varus_valgus_angle']):.1f}°")
print(f"Range: [{np.min(mech_axis_dyn['varus_valgus_angle']):.1f}°, {np.max(mech_axis_dyn['varus_valgus_angle']):.1f}°]")
print(f"Mean knee offset: {np.mean(mech_axis_dyn['knee_offset']):.1f} ± {np.std(mech_axis_dyn['knee_offset']):.1f} mm")
print(f"Range: [{np.min(mech_axis_dyn['knee_offset']):.1f}, {np.max(mech_axis_dyn['knee_offset']):.1f}] mm")

Loading dynamic rotation trial...


FileNotFoundError: [Errno 2] No such file or directory: './data/dynamic__trial_rotation_1.csv'

## 10. Visualize Dynamic Trial

In [ ]:
# Plot time series of angles
fig3 = plot_angle_time_series(
    mech_axis_dyn,
    fps=100,
    title="Dynamic Rotation Trial - Joint Angles Over Time"
)
plt.show()

# Plot 3D skeleton at different time points
frames_to_plot = [0, len(knee_center_dyn)//4, len(knee_center_dyn)//2, 3*len(knee_center_dyn)//4]

for i, frame_idx in enumerate(frames_to_plot):
    fig = plot_3d_skeleton(
        hip_center_dyn,
        knee_center_dyn,
        ankle_center_dyn,
        mechanical_axis_info=mech_axis_dyn,
        frame_idx=frame_idx,
        title=f"Dynamic Rotation - Frame {frame_idx} ({frame_idx/100:.2f}s)"
    )
    plt.show()

NameError: name 'mech_axis_dyn' is not defined

## 11. Compare Multiple Dynamic Trials

In [ ]:
# Load and analyze multiple dynamic trials
dynamic_files = [
    'dynamic__trial_rotation_1.csv',
    'dynamic__trial_rotation_2.csv',
    'dynamic__trial_bending_varus_motion_1.csv'
]

trial_names = ['Rotation 1', 'Rotation 2', 'Varus Bending']
results = []

for filename, trial_name in zip(dynamic_files, trial_names):
    try:
        print(f"\nAnalyzing {trial_name}...")
        trial = load_mocap_csv(DATA_DIR + filename)
        
        # Extract markers
        FH_t = extract_marker_trajectory(trial, 'FH')
        MFEC_t = extract_marker_trajectory(trial, 'MFEC')
        LFEC_t = extract_marker_trajectory(trial, 'LFEC')
        MTC_t = extract_marker_trajectory(trial, 'MTC')
        LTC_t = extract_marker_trajectory(trial, 'LTC')
        MM_t = extract_marker_trajectory(trial, 'MM')
        LM_t = extract_marker_trajectory(trial, 'LM')
        
        # Calculate centers
        hip_c = calculate_hip_center(FH_t)
        knee_c = calculate_knee_center(MTC_t, LTC_t, MFEC_t, LFEC_t)
        ankle_c = calculate_ankle_center(MM_t, LM_t)
        
        # Calculate mechanical axis
        mech_axis = calculate_mechanical_axis(hip_c, knee_c, ankle_c)
        
        results.append({
            'name': trial_name,
            'mech_axis': mech_axis,
            'hip': hip_c,
            'knee': knee_c,
            'ankle': ankle_c
        })
        
        print(f"  Mean angle: {np.mean(mech_axis['varus_valgus_angle']):.1f}° ± {np.std(mech_axis['varus_valgus_angle']):.1f}°")
        print(f"  Mean offset: {np.mean(mech_axis['knee_offset']):.1f} ± {np.std(mech_axis['knee_offset']):.1f} mm")
        
    except Exception as e:
        print(f"  Error processing {filename}: {str(e)}")

print("\nAnalysis complete!")


Analyzing Rotation 1...
  Error processing dynamic__trial_rotation_1.csv: [Errno 2] No such file or directory: './data/dynamic__trial_rotation_1.csv'

Analyzing Rotation 2...
  Error processing dynamic__trial_rotation_2.csv: [Errno 2] No such file or directory: './data/dynamic__trial_rotation_2.csv'

Analyzing Varus Bending...
  Error processing dynamic__trial_bending_varus_motion_1.csv: [Errno 2] No such file or directory: './data/dynamic__trial_bending_varus_motion_1.csv'

Analysis complete!


## 12. Comparative Visualization

In [ ]:
# Compare varus/valgus angles across trials
if len(results) > 0:
    fig, axes = plt.subplots(len(results), 2, figsize=(14, 4*len(results)))
    
    if len(results) == 1:
        axes = axes.reshape(1, -1)
    
    for i, result in enumerate(results):
        mech = result['mech_axis']
        time = np.arange(len(mech['varus_valgus_angle'])) / 100
        
        # Angle plot
        axes[i, 0].plot(time, mech['varus_valgus_angle'], 'b-', linewidth=2)
        axes[i, 0].axhline(y=180, color='r', linestyle='--', alpha=0.5)
        axes[i, 0].set_ylabel('Angle (°)', fontsize=11)
        axes[i, 0].set_title(f"{result['name']} - Varus/Valgus Angle", fontweight='bold')
        axes[i, 0].grid(True, alpha=0.3)
        
        # Offset plot
        axes[i, 1].plot(time, mech['knee_offset'], 'g-', linewidth=2)
        axes[i, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[i, 1].set_ylabel('Offset (mm)', fontsize=11)
        axes[i, 1].set_title(f"{result['name']} - Knee Offset", fontweight='bold')
        axes[i, 1].grid(True, alpha=0.3)
        
        if i == len(results) - 1:
            axes[i, 0].set_xlabel('Time (s)', fontsize=11)
            axes[i, 1].set_xlabel('Time (s)', fontsize=11)
    
    plt.tight_layout()
    plt.show()

## 13. Summary Statistics

In [ ]:
# Create summary table
if len(results) > 0:
    summary_data = []
    
    for result in results:
        mech = result['mech_axis']
        summary_data.append({
            'Trial': result['name'],
            'Mean Angle (°)': f"{np.mean(mech['varus_valgus_angle']):.2f}",
            'Std Angle (°)': f"{np.std(mech['varus_valgus_angle']):.2f}",
            'Min Angle (°)': f"{np.min(mech['varus_valgus_angle']):.2f}",
            'Max Angle (°)': f"{np.max(mech['varus_valgus_angle']):.2f}",
            'Mean Offset (mm)': f"{np.mean(mech['knee_offset']):.2f}",
            'Std Offset (mm)': f"{np.std(mech['knee_offset']):.2f}",
            'Min Offset (mm)': f"{np.min(mech['knee_offset']):.2f}",
            'Max Offset (mm)': f"{np.max(mech['knee_offset']):.2f}"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n" + "="*100)
    print("SUMMARY STATISTICS - ALL TRIALS")
    print("="*100)
    print(summary_df.to_string(index=False))
    print("\n" + "="*100)

## 14. Export Results

In [ ]:
# Save summary statistics to CSV
if len(results) > 0:
    output_file = '/mnt/user-data/outputs/mechanical_axis_summary.csv'
    summary_df.to_csv(output_file, index=False)
    print(f"Summary statistics saved to: {output_file}")

# Save detailed results for each trial
for result in results:
    trial_name = result['name'].replace(' ', '_').lower()
    mech = result['mech_axis']
    
    detailed_df = pd.DataFrame({
        'Frame': np.arange(len(mech['varus_valgus_angle'])),
        'Time_s': np.arange(len(mech['varus_valgus_angle'])) / 100,
        'Varus_Valgus_Angle_deg': mech['varus_valgus_angle'],
        'Knee_Offset_mm': mech['knee_offset'],
        'Axis_Length_mm': mech['axis_length']
    })
    
    output_file = f'/mnt/user-data/outputs/{trial_name}_detailed.csv'
    detailed_df.to_csv(output_file, index=False)
    print(f"Detailed results for {result['name']} saved to: {output_file}")

print("\nAll results exported successfully!")


All results exported successfully!


## Interpretation Guide

### Mechanical Axis
The mechanical axis is the line connecting the center of the femoral head (hip) to the center of the ankle. In normal alignment, this line should pass through or very close to the center of the knee.

### Key Metrics

1. **Varus/Valgus Angle**:
   - **180°**: Perfect alignment (femur and tibia are straight)
   - **>180°**: Valgus deformity (knock-knee)
   - **<180°**: Varus deformity (bow-legged)
   - **Normal range**: 175-185°

2. **Knee Offset**:
   - Perpendicular distance from knee center to mechanical axis
   - **Normal**: <20mm
   - **Abnormal**: >20mm indicates significant malalignment

3. **Clinical Significance**:
   - Malalignment increases stress on knee cartilage
   - Can lead to osteoarthritis progression
   - Important for surgical planning (e.g., osteotomy, arthroplasty)

### Dynamic Analysis
- Observe how alignment changes during movement
- Larger variations may indicate instability
- Compare to normative data for your specific motion tasks